# ViSceT5 — Finetune + Ablation · nhánh `exp/ablation-drop-ocr` · Colab

Chạy **đúng quy trình của repo**: notebook chỉ đặt biến môi trường rồi gọi `bash run_all_colab.sh finetune`
(clone nhánh → `setup.sh` → `run_pipeline.py`: chuẩn bị dữ liệu → khởi tạo model → `training/finetune.py configs/finetune.yaml` → upload HF).
Mọi thông số huấn luyện lấy từ **`configs/finetune.yaml`** (epoch, batch, LR, scheduler, label smoothing, beam, …); notebook **không** ghi đè.

Mỗi lần chạy = **một cấu hình**, chọn bằng 3 công tắc bật/tắt module (Visual Search luôn tắt):

| Công tắc | `True` | `False` |
|---|---|---|
| `USE_IMAGE` | có đặc trưng ảnh (CLIP) | **bỏ hẳn ảnh**: chỉ câu hỏi + OCR (CLIP không chạy) |
| `USE_QAVIT` | đưa câu hỏi vào CLIP (QA-ViT) | CLIP trích xuất ảnh thường (không câu hỏi) |
| `USE_SCESPAVIS` | OCR qua SceSpaVis (ConsFormer + char + box + det/rec) | OCR text-only |
| `USE_OCRAUG` | nối thêm bản OCR đã sửa (OCR-Aug) | chỉ OCR gốc |

**ViSceT5 (kết quả chính):** `USE_IMAGE=True, USE_QAVIT=False, USE_SCESPAVIS=True, USE_OCRAUG=True`. Tắt `USE_IMAGE` để đo mô hình chỉ dùng câu hỏi + OCR (`USE_QAVIT` khi đó phải là `False`). Bảng ablation: chạy lại với các tổ hợp khác (tắt hết = baseline).

## 1. Lấy repo (nhánh `exp/ablation-drop-ocr`)

In [ ]:
import os
BRANCH = "exp/ablation-drop-ocr"
if not os.path.exists("/content/ViSceT5"):
    !git clone -b {BRANCH} https://github.com/Kussssssss/ViSceT5.git /content/ViSceT5
%cd /content/ViSceT5
!git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
!git log --oneline -3
assert "_specials" in open("data/collator.py", encoding="utf-8").read(), "Thiếu bản sửa OCR-Aug — kiểm tra lại nhánh!"
assert "_drop_unused_modules" in open("models/openvivqa_model.py", encoding="utf-8").read(), "Code cũ: module không dùng vẫn nằm trong kiến trúc — kiểm tra lại nhánh!"

## 2. Hugging Face token (để `finetune.py` tự push checkpoint mỗi epoch, tự resume khi Colab ngắt, và upload cuối)
Thêm secret `HF_TOKEN` (tab 🔑, bật *Notebook access*), **hoặc** chạy cell dưới rồi dán token.

In [ ]:
import os
try:
    from google.colab import userdata
    _t = userdata.get("HF_TOKEN")
    if _t:
        os.environ["HF_TOKEN"] = _t
except Exception as e:
    print("Colab Secrets:", e)
if not os.environ.get("HF_TOKEN"):
    from huggingface_hub import notebook_login
    notebook_login()
else:
    print("✅ Đã có HF_TOKEN từ Colab Secrets.")

## 3. Cấu hình — bật/tắt module

In [ ]:
USE_IMAGE     = True
USE_QAVIT     = False
USE_SCESPAVIS = True
USE_OCRAUG    = True
DATASET       = "ViTextVQA"
RUN_TAG       = "r3"

import os
from huggingface_hub import get_token, HfApi
_tok = os.environ.get("HF_TOKEN") or get_token()
if _tok:
    os.environ["HF_TOKEN"] = _tok
HF_USER = "Kus669"
if _tok:
    try:
        HF_USER = HfApi(token=_tok).whoami().get("name") or HF_USER
    except Exception as e:
        print("whoami lỗi:", e)
else:
    print("⚠️ CHƯA có HF_TOKEN: checkpoint chỉ nằm trên Colab, mất khi ngắt kết nối!")

assert USE_IMAGE or not USE_QAVIT, "QA-ViT cần ảnh: khi USE_IMAGE=False phải đặt USE_QAVIT=False"
RUN_NAME = "-".join(m for m, on in (("noimg", not USE_IMAGE), ("qavit", USE_QAVIT), ("scespavis", USE_SCESPAVIS),
                                    ("ocraug", USE_OCRAUG)) if on) or "base"
OUTPUT_DIR = f"/content/ViSceT5/output/abl_{DATASET}_{RUN_NAME}"
HF_REPO = f"{HF_USER}/ViSceT5-abl-{DATASET.lower()}-{RUN_NAME}-{RUN_TAG}"
_b = lambda v: "true" if v else "false"

os.environ.update({
    "GIT_BRANCH":           "exp/ablation-drop-ocr",
    "ABLATION_USE_IMAGE":   _b(USE_IMAGE),
    "ABLATION_USE_QACLIP":  _b(USE_QAVIT),
    "ABLATION_USE_VS":      "false",
    "ABLATION_USE_OCR":     _b(USE_SCESPAVIS),
    "ABLATION_USE_OCR_AUG": _b(USE_OCRAUG),
    "DATASET_NAME":         DATASET,
    "OUTPUT_DIR":           OUTPUT_DIR,
    "HF_REPO":              HF_REPO,
})
_main = USE_IMAGE and (not USE_QAVIT) and USE_SCESPAVIS and USE_OCRAUG
print(f"Cấu hình: Ảnh={USE_IMAGE} | QA-ViT={USE_QAVIT} | SceSpaVis={USE_SCESPAVIS} | OCR-Aug={USE_OCRAUG}"
      f"{'   ← ViSceT5 (kết quả chính)' if _main else ''}")
print(f"RUN_NAME={RUN_NAME}\nOUTPUT_DIR={OUTPUT_DIR}\nHF_REPO={HF_REPO}")
print("Thông số huấn luyện: lấy từ configs/finetune.yaml")

## 4. (Tuỳ chọn) Mock test — vài bước để bắt lỗi trước khi chạy thật
Dùng chế độ `mock` có sẵn của `run_all_colab.sh` (`--smoke_test`); không lưu checkpoint, không resume từ HF.

In [ ]:
!HF_REPO= OUTPUT_DIR={OUTPUT_DIR}_mock bash run_all_colab.sh finetune mock

## 5. Finetune (từ đầu, không pretrain) — quy trình chuẩn của repo

In [ ]:
!bash run_all_colab.sh finetune

## 6. Dự đoán dev + test (`STAGE=predict` của `run_pipeline.py`) và chấm điểm
`predict.py` in EM/F1 trên dev và ghi file nộp cho test. File được chép vào `OUTPUT_DIR` để không đè giữa các cấu hình.
Nếu có file đáp án test, đặt `TEST_GT_JSON` để `training/eval_submission.py` chấm EM, F1, BLEU, CIDEr.

In [ ]:
import os, shutil, glob
TEST_GT_JSON = ""

!STAGE=predict PREDICT_SPLIT=both PREDICT_DATASET={DATASET} PREDICT_CKPT_DIR={OUTPUT_DIR} HF_REPO= bash run_all_colab.sh

tag = "" if DATASET == "ViTextVQA" else f"{DATASET}_"
for sp in ("dev", "test"):
    src = f"/content/ViSceT5/output/submission_{tag}{sp}.csv"
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(OUTPUT_DIR, os.path.basename(src)))
        print("→", os.path.join(OUTPUT_DIR, os.path.basename(src)))
pred = os.path.join(OUTPUT_DIR, f"submission_{tag}test.csv")
if TEST_GT_JSON and os.path.exists(TEST_GT_JSON) and os.path.exists(pred):
    !python -m training.eval_submission --gt {TEST_GT_JSON} --pred {pred} --out {OUTPUT_DIR}/eval_test.json
else:
    print("ℹ️ Chưa đặt TEST_GT_JSON → chưa chấm test.")

## 7. Tổng hợp kết quả mọi cấu hình đã chạy

In [ ]:
import json, os, glob
import pandas as pd
rows = []
for od in sorted(glob.glob(f"/content/ViSceT5/output/abl_{DATASET}_*")):
    name = od.split(f"abl_{DATASET}_")[-1]
    mods = set(name.split("-"))
    r = {"run": name, "Ảnh": "noimg" not in mods, "QA-ViT": "qavit" in mods,
         "SceSpaVis": "scespavis" in mods, "OCR-Aug": "ocraug" in mods}
    st = sorted(glob.glob(os.path.join(od, "checkpoint-*", "trainer_state.json")),
                key=lambda p: int(p.split("checkpoint-")[-1].split(os.sep)[0]))
    if st:
        key = "eval_cider" if DATASET == "OpenViVQA" else "eval_em"
        evs = [h for h in json.load(open(st[-1])).get("log_history", []) if key in h]
        if evs:
            best = max(evs, key=lambda h: h[key])
            r.update({"val_EM": round(100 * best.get("eval_em", float("nan")), 2), "val_F1": round(100 * best.get("eval_f1", 0), 2),
                      "val_CIDEr": round(best.get("eval_cider", float("nan")), 4), "best_epoch": best.get("epoch")})
    ev = os.path.join(od, "eval_test.json")
    if os.path.exists(ev):
        r.update({f"test_{k}": v for k, v in json.load(open(ev)).items() if not isinstance(v, (dict, list))})
    rows.append(r)
df = pd.DataFrame(rows)
print(df.to_string(index=False) if len(df) else "Chưa có cấu hình nào chạy xong.")